# FL Compression Study — 1-seed sweep
Runs every config once (seed 0, 100 rounds). Designed for Colab with Drive mount.

**Configs included:**
- FP32 baseline
- Quant 8-bit
- Quant 4-bit (expected to collapse — kept for completeness)
- USNR-RMS α=0.2
- FP32 + cosine LR decay
- Quant 8-bit + cosine LR decay
- SZ Schedule A (eb 0.001→0.01 at round 50)
- SZ Schedule B (eb 0.001→0.005→0.01)
- SZ Schedule C (plateau-triggered)

**Output:** `results/resnet20_cifar10_1seed.csv`  
Runs resume automatically from checkpoint if Colab disconnects.

## Cell 1 — Install dependencies (run once, then restart runtime)

In [ ]:
# Install order matters: numpy first to avoid binary conflict
!pip install -q 'numpy<2.0'
!pip install -q 'flwr[simulation]==1.9.0' ray pandas seaborn
!pip install -q 'protobuf>=4.23' --force-reinstall
!pip install -q pysz
print('Done. Now restart the runtime (Runtime → Restart runtime), then run Cell 2.')

## Cell 2 — Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_DIR = '/content/fl-compression-study'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/YOUR_USERNAME/fl-compression-study.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Link results and checkpoints to Drive so they survive disconnects
DRIVE_BASE = '/content/drive/MyDrive/fl_results'
os.makedirs(f'{DRIVE_BASE}/results',     exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)

for folder in ['results', 'checkpoints']:
    local = f'{REPO_DIR}/{folder}'
    drive_path = f'{DRIVE_BASE}/{folder}'
    if not os.path.islink(local):
        if os.path.exists(local):
            import shutil; shutil.rmtree(local)
        os.symlink(drive_path, local)

print('Repo ready. Results will be saved to Drive.')
print('Working dir:', os.getcwd())

## Cell 3 — Run all configs (seed 0, 100 rounds each)

In [ ]:
import os, subprocess, sys
from datetime import datetime

REPO_DIR = '/content/fl-compression-study'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

PYTHON   = sys.executable
RUN_PY   = f'{REPO_DIR}/fl-flower/run.py'
OUTPUT   = f'{REPO_DIR}/results/resnet20_cifar10_1seed.csv'
ROUNDS   = 100
SEED     = 0

BASE_FLAGS = [
    '--dataset',     'cifar10',
    '--rounds',      str(ROUNDS),
    '--num-clients', '10',
    '--alpha',       '0.5',
    '--local-epochs','2',
    '--seed',        str(SEED),
    '--output',      OUTPUT,
]

# (name, extra flags)
CONFIGS = [
    ('fp32_baseline',
     ['--compressor', 'none']),

    ('quant_8bit',
     ['--compressor', 'quantization', '--bits', '8']),

    ('quant_4bit',
     ['--compressor', 'quantization', '--bits', '4']),

    ('usnr_a02',
     ['--compressor', 'sz_usnr_rms',
      '--usnr-alpha', '0.2',
      '--usnr-eb-min', '1e-6',
      '--usnr-eb-max', '1.0']),

    ('fp32_cosine',
     ['--compressor', 'none', '--lr-decay']),

    ('quant8_cosine',
     ['--compressor', 'quantization', '--bits', '8', '--lr-decay']),

    ('sz_schedule_a',
     ['--compressor', 'sz', '--schedule', 'A']),

    ('sz_schedule_b',
     ['--compressor', 'sz', '--schedule', 'B']),

    ('sz_schedule_c',
     ['--compressor', 'sz', '--schedule', 'C']),
]

total = len(CONFIGS)
for i, (name, extra) in enumerate(CONFIGS, 1):
    print(f'\n{"="*60}')
    print(f'[{i}/{total}]  {name}  |  {datetime.now().strftime("%H:%M:%S")}')
    print(f'{"="*60}')
    cmd = [PYTHON, RUN_PY] + BASE_FLAGS + extra
    result = subprocess.run(cmd, cwd=REPO_DIR)
    if result.returncode != 0:
        print(f'  *** {name} exited with code {result.returncode} — continuing to next config ***')
    else:
        print(f'  {name} done at {datetime.now().strftime("%H:%M:%S")}')

print('\nAll configs finished.')

## Cell 4 — Quick results summary

In [ ]:
import pandas as pd, numpy as np

OUTPUT = '/content/fl-compression-study/results/resnet20_cifar10_1seed.csv'

try:
    df = pd.read_csv(OUTPUT)
except Exception:
    # handle mixed-column CSVs written by old + new strategy
    with open(OUTPUT) as f:
        lines = f.readlines()
    hdr = lines[0].strip().split(',')
    rows = [dict(zip(hdr, l.strip().split(','))) for l in lines[1:]
            if len(l.strip().split(',')) == len(hdr)]
    df = pd.DataFrame(rows)
    for c in ['round','val_accuracy','bytes_sent','compression_ratio']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

fp32_bytes = df[(df['compressor']=='fp32_baseline')&(df['round']>0)]['bytes_sent'].mean()

rows_out = []
for comp, grp in df.groupby('compressor'):
    peak     = grp['val_accuracy'].max()
    latest   = grp.nlargest(1,'round')['val_accuracy'].values[0]
    max_r    = int(grp['round'].max())
    ratio    = grp[grp['round']>0]['compression_ratio'].mean()
    mb       = grp[grp['round']>0]['bytes_sent'].mean()/10/1e6
    rows_out.append({'config':comp,'max_round':max_r,'peak_acc':round(peak,2),
                     'latest_acc':round(latest,2),'ratio':round(ratio,2),
                     'MB_per_round_per_client':round(mb,3)})

summary = pd.DataFrame(rows_out).sort_values('peak_acc', ascending=False)
print('=== 1-seed sweep summary ===')
print(summary.to_string(index=False))

## Cell 5 — Convergence plot

In [ ]:
import matplotlib.pyplot as plt

PALETTE = {
    'fp32_baseline':  '#2c7bb6',
    'quant_8bit':     '#d7191c',
    'quant_4bit':     '#fdae61',
    'usnr_a02':       '#7b2d8b',
    'fp32_cosine':    '#74add1',
    'quant8_cosine':  '#f46d43',
    'sz_schedule_a':  '#1a9641',
    'sz_schedule_b':  '#66bd63',
    'sz_schedule_c':  '#006837',
}

fig, ax = plt.subplots(figsize=(12, 6))
for comp, grp in df.groupby('compressor'):
    grp = grp.sort_values('round')
    rm  = grp['val_accuracy'].rolling(5, min_periods=1).mean()
    ax.plot(grp['round'], rm,
            color=PALETTE.get(comp, '#888'),
            lw=2, label=comp)

ax.set_xlabel('Communication round', fontsize=12)
ax.set_ylabel('Validation accuracy (%)', fontsize=12)
ax.set_title('1-seed sweep — convergence (seed 0, 100 rounds)', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('/content/fl-compression-study/results/1seed_convergence.png', dpi=150)
plt.show()
print('Saved 1seed_convergence.png')